# Code Graph Workbench

Ask the AI sidebar about the codebase; evidence tools paint these panels.
Panels: analyst (left), graph (center), inspector (bottom). Spec:
`docs/superpowers/specs/2026-06-09-code-graph-workbench-app-design.md`.

In [ ]:
// wb-graph placeholder - bound to `subgraph` + `scorecard` evidence ports.
Deno.jupyter.html`<div data-wb="graph"><h3>Graph</h3><p>no evidence pushed this turn</p></div>`;

In [ ]:
// wb-analyst placeholder - bound to `analyst_rows` evidence port.
Deno.jupyter.html`<div data-wb="analyst"><h3>Analyst</h3><p>no evidence pushed this turn</p></div>`;

In [ ]:
// wb-inspector placeholder - bound to `scorecard` + `cochange` evidence ports.
Deno.jupyter.html`<div data-wb="inspector"><h3>Inspector</h3><p>no evidence pushed this turn</p></div>`;

In [ ]:
// wb-status placeholder - status rail; cochange source port declared here.
Deno.jupyter.html`<div data-wb="status"><h3>Status</h3><p>no evidence pushed this turn</p></div>`;

In [ ]:
# SPUR datasource setup cell v1
# This cell is managed by SPUR. Re-run it after datasource changes.
import duckdb

_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
duckdb.sql(f"LOAD '{_SPUR_DUCKDB_EXTENSION_SQL}'")

duckdb.sql("CREATE OR REPLACE VIEW \"5b7d070b62a4da1a4f9d4bbb6b2d6e0ed2e2af2455b3935917246212780ea94e\" AS SELECT * FROM read_parquet('/Volumes/Projects/spur/.spur/graph/5b7d070b62a4da1a4f9d4bbb6b2d6e0ed2e2af2455b3935917246212780ea94e.parquet')")


# RSS / RSSHub Subscription UI Brief

Surface: notebook datasource onboarding and subscription flow for RSS/RSSHub sources.

Audience: users who want to add a direct RSS feed, subscribe through an RSSHub route, or discover sources by keyword without writing SQL first.

Primary job: let a user move from unknown source input to validated feed preview to subscribed source.

Technical grounding:
- Existing gateway exposes `rss_routes`, `rss_feed(url)`, and `rss_entries(url)`.
- Existing UI has an Add RSS / RSSHub wizard step.
- Research doc defines three entry paths: direct URL, `rsshub://` route, and keyword discovery.

Design direction: quiet operational UI, dense enough for browsing many routes, with a clear right-side subscription preview and explicit health/validation states.

Assumptions to validate later:
- Subscription persistence is a near-term product layer above the current datasource/table-functions capability.
- `rsshub://` remains the canonical user-facing route format.
- Keyword discovery can initially search `rss_routes` plus known feed metadata before a broader feed directory exists.

# RSS / RSSHub UI Plan

1. Build a single high-fidelity HTML prototype for the RSS subscription workflow.
2. Represent all three sourcing paths: direct feed URL, RSSHub route browser, and keyword discovery.
3. Include route parameterization, feed preview, subscription options, and health/status states.
4. Keep the visual style aligned with Jute/SPUR: compact panels, restrained color, tables/lists optimized for scanning.
5. Critique for clarity, density, responsive behavior, and implementation fit against the existing RSS datasource primitives.

In [2]:
// open-design artifact: RSS / RSSHub subscription workflow
const html = String.raw`<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
:root{
  --bg:#f6f5f2;
  --surface:#ffffff;
  --ink:#171717;
  --muted:#676b73;
  --line:#d9d7d2;
  --line-strong:#bdb8ad;
  --teal:#11756b;
  --teal-soft:#d9eeea;
  --amber:#b36b00;
  --amber-soft:#fff0ce;
  --red:#b42318;
  --red-soft:#ffe2de;
  --green:#16794c;
  --green-soft:#dcf3e8;
  --blue:#315f9c;
  --blue-soft:#dde9f7;
  --shadow:0 16px 34px rgba(42,37,30,.12);
}
*{box-sizing:border-box}
html,body{margin:0;min-height:100%;background:var(--bg);color:var(--ink);font-family:Inter,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;letter-spacing:0}
button,input,select{font:inherit}
.rss-app{min-height:100vh;padding:18px;background:linear-gradient(180deg,#faf9f6 0,#f0eee8 100%)}
.shell{max-width:1420px;margin:0 auto;background:var(--surface);border:1px solid var(--line);box-shadow:var(--shadow);border-radius:8px;overflow:hidden}
.topbar{height:56px;display:flex;align-items:center;gap:14px;padding:0 16px;border-bottom:1px solid var(--line);background:#fbfaf8}
.brand{display:flex;align-items:center;gap:10px;min-width:240px}
.mark{width:28px;height:28px;border:1px solid #222;border-radius:6px;display:grid;place-items:center;background:#f7c95f;color:#111;font-weight:800;font-size:14px}
.brand-title{font-weight:700;font-size:14px;line-height:1}
.brand-sub{font-size:12px;color:var(--muted);margin-top:3px}
.tabs{display:flex;align-items:center;gap:4px;border:1px solid var(--line);border-radius:7px;background:#fff;padding:3px}
.tab{height:30px;border:0;background:transparent;border-radius:5px;padding:0 10px;color:#555;font-size:12px;cursor:pointer}
.tab.active{background:#171717;color:#fff}
.top-actions{margin-left:auto;display:flex;align-items:center;gap:8px}
.status-pill{height:28px;display:inline-flex;align-items:center;gap:7px;border:1px solid var(--line);border-radius:999px;padding:0 10px;font-size:12px;color:#555;background:#fff}
.dot{width:7px;height:7px;border-radius:50%;background:var(--green)}
.primary{height:32px;border:1px solid #111;background:#171717;color:#fff;border-radius:6px;padding:0 12px;font-size:12px;font-weight:650;cursor:pointer}
.secondary{height:32px;border:1px solid var(--line-strong);background:#fff;color:#1f1f1f;border-radius:6px;padding:0 12px;font-size:12px;font-weight:600;cursor:pointer}
.main{display:grid;grid-template-columns:330px minmax(430px,1fr) 360px;min-height:760px}
.panel{border-right:1px solid var(--line);min-width:0;background:#fff}
.panel:last-child{border-right:0}
.panel-head{height:62px;padding:12px 14px;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;gap:12px;background:#fbfaf8}
.panel-title{font-size:13px;font-weight:750}
.panel-note{font-size:11px;color:var(--muted);margin-top:3px}
.count{font-variant-numeric:tabular-nums;font-size:11px;color:#555;border:1px solid var(--line);border-radius:999px;padding:4px 8px;background:#fff}
.source-input{padding:14px;border-bottom:1px solid var(--line);background:#fff}
.input-wrap{display:flex;gap:8px;align-items:center}
.input-wrap input{height:38px;min-width:0;flex:1;border:1px solid var(--line-strong);border-radius:6px;padding:0 10px;font-size:13px;outline:none;background:#fff;color:#111}
.input-wrap input:focus{border-color:var(--teal);box-shadow:0 0 0 3px var(--teal-soft)}
.detect-row{display:grid;grid-template-columns:repeat(3,1fr);gap:6px;margin-top:10px}
.detect{border:1px solid var(--line);border-radius:6px;padding:8px;background:#fbfaf8;min-height:62px}
.detect.active{border-color:var(--teal);background:var(--teal-soft)}
.detect b{display:block;font-size:11px;margin-bottom:4px}.detect span{display:block;font-size:11px;color:#5b5b5b;line-height:1.25}
.categories{padding:10px 10px 0;display:flex;gap:6px;flex-wrap:wrap}
.chip{height:26px;border:1px solid var(--line);border-radius:999px;background:#fff;padding:0 9px;color:#4b4b4b;font-size:11px;cursor:pointer}.chip.active{border-color:#111;background:#171717;color:#fff}
.route-list{padding:10px;display:flex;flex-direction:column;gap:8px;max-height:535px;overflow:auto}
.route-card{border:1px solid var(--line);border-radius:8px;background:#fff;padding:10px;text-align:left;cursor:pointer}.route-card.active{border-color:var(--teal);box-shadow:inset 3px 0 0 var(--teal)}.route-card:hover{border-color:#aaa}
.route-top{display:flex;align-items:flex-start;justify-content:space-between;gap:10px}.route-name{font-size:13px;font-weight:750}.route-path{font-family:"SF Mono",Consolas,monospace;font-size:11px;color:#555;margin-top:4px;word-break:break-all}.route-meta{display:flex;gap:6px;flex-wrap:wrap;margin-top:8px}.tag{font-size:10px;border:1px solid var(--line);border-radius:999px;padding:3px 6px;color:#555;background:#fafafa}.heat{font-variant-numeric:tabular-nums;color:var(--amber);background:var(--amber-soft);border-color:#e4c778}
.workflow{display:grid;grid-template-rows:auto 1fr;background:#fdfcf9}
.steps{height:62px;border-bottom:1px solid var(--line);display:grid;grid-template-columns:repeat(4,1fr);background:#fbfaf8}
.step{display:flex;align-items:center;gap:9px;padding:0 14px;border-right:1px solid var(--line);min-width:0}.step:last-child{border-right:0}.step-num{width:24px;height:24px;border-radius:6px;display:grid;place-items:center;background:#eee;border:1px solid var(--line);font-size:11px;font-weight:750}.step.active .step-num{background:#171717;color:#fff;border-color:#171717}.step.done .step-num{background:var(--green-soft);color:var(--green);border-color:#a6d8c0}.step-text{min-width:0}.step-label{font-size:11px;font-weight:750;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.step-detail{font-size:10px;color:var(--muted);margin-top:2px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.canvas{padding:16px;display:grid;grid-template-columns:minmax(0,1.08fr) minmax(300px,.92fr);gap:14px;align-content:start}.section{border:1px solid var(--line);border-radius:8px;background:#fff;overflow:hidden}.section-head{padding:12px 14px;border-bottom:1px solid var(--line);background:#fbfaf8;display:flex;align-items:center;justify-content:space-between;gap:12px}.section-title{font-size:13px;font-weight:750}.section-body{padding:14px}.field{display:grid;gap:6px;margin-bottom:12px}.field label{font-size:11px;font-weight:700;color:#4b4b4b}.field input,.field select{height:36px;border:1px solid var(--line-strong);border-radius:6px;padding:0 9px;background:#fff;outline:none;color:#111;font-size:13px}.field input:focus,.field select:focus{border-color:var(--teal);box-shadow:0 0 0 3px var(--teal-soft)}.mono{font-family:"SF Mono",Consolas,monospace}.url-box{border:1px solid var(--line);border-radius:6px;background:#f7f6f2;padding:10px;font-family:"SF Mono",Consolas,monospace;font-size:12px;line-height:1.45;word-break:break-all}.validation{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin-top:12px}.metric{border:1px solid var(--line);border-radius:7px;padding:9px;background:#fbfaf8}.metric b{display:block;font-size:13px;font-variant-numeric:tabular-nums}.metric span{display:block;font-size:10px;color:var(--muted);margin-top:3px}.ok{color:var(--green)}.warn{color:var(--amber)}
.preview-card{border:1px solid var(--line);border-radius:8px;overflow:hidden}.feed-head{display:flex;gap:11px;padding:12px;border-bottom:1px solid var(--line);background:#fff}.favicon{width:42px;height:42px;border-radius:8px;background:linear-gradient(135deg,#f7c95f,#55b9a9);display:grid;place-items:center;font-weight:800}.feed-title{font-size:14px;font-weight:800}.feed-desc{font-size:12px;color:var(--muted);line-height:1.35;margin-top:4px}.entry{padding:10px 12px;border-top:1px solid var(--line)}.entry:first-child{border-top:0}.entry-title{font-size:12px;font-weight:750}.entry-meta{font-size:11px;color:var(--muted);margin-top:4px}.mapping-grid{display:grid;grid-template-columns:1fr 1fr;gap:8px}.map-item{border:1px solid var(--line);border-radius:7px;padding:9px;background:#fbfaf8}.map-label{font-size:10px;color:#666;text-transform:uppercase}.map-value{font-size:12px;font-weight:750;margin-top:4px}
.subscribe{background:#fbfaf8}.subscribe .panel-head{height:62px}.sub-body{padding:14px}.summary{border:1px solid var(--line);border-radius:8px;background:#fff;overflow:hidden;margin-bottom:12px}.summary-row{display:flex;justify-content:space-between;gap:12px;padding:10px 12px;border-top:1px solid var(--line);font-size:12px}.summary-row:first-child{border-top:0}.summary-row span:first-child{color:var(--muted)}.summary-row span:last-child{text-align:right;font-weight:650}.toggle-list{display:grid;gap:8px;margin:12px 0}.toggle{display:flex;align-items:center;justify-content:space-between;gap:10px;border:1px solid var(--line);border-radius:7px;padding:10px;background:#fff}.toggle-text b{display:block;font-size:12px}.toggle-text span{display:block;font-size:11px;color:var(--muted);margin-top:3px}.switch{width:36px;height:20px;border-radius:999px;background:#d8d5ce;position:relative;flex:0 0 auto}.switch:after{content:"";position:absolute;width:16px;height:16px;top:2px;left:2px;border-radius:50%;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.2)}.switch.on{background:var(--teal)}.switch.on:after{left:18px}.health{border:1px solid #b8ddcc;background:var(--green-soft);color:#125b3b;border-radius:8px;padding:10px;font-size:12px;line-height:1.4}.health.warn-state{border-color:#e6c16c;background:var(--amber-soft);color:#7a4800}.action-stack{display:grid;gap:8px;margin-top:12px}.action-stack button{height:38px}.activity{margin-top:14px;border-top:1px solid var(--line);padding-top:12px}.activity h4{font-size:12px;margin:0 0 8px}.event{display:grid;grid-template-columns:18px 1fr;gap:8px;margin-top:8px}.event-dot{width:8px;height:8px;border-radius:50%;background:var(--green);margin-top:5px}.event b{display:block;font-size:12px}.event span{display:block;font-size:11px;color:var(--muted);margin-top:2px;line-height:1.35}.implementation{margin-top:14px;border:1px solid var(--line);border-radius:8px;background:#fff;padding:12px}.implementation h4{font-size:12px;margin:0 0 8px}.impl-row{display:grid;grid-template-columns:104px 1fr;gap:8px;font-size:11px;padding:6px 0;border-top:1px solid #eee}.impl-row:first-of-type{border-top:0}.impl-row b{color:#444}.impl-row code{font-family:"SF Mono",Consolas,monospace;color:#222;word-break:break-word}
@media (max-width:1100px){.main{grid-template-columns:300px 1fr}.subscribe{grid-column:1 / -1;border-top:1px solid var(--line)}.canvas{grid-template-columns:1fr 1fr}.sub-body{display:grid;grid-template-columns:1fr 1fr;gap:14px}.implementation{margin-top:0}}
@media (max-width:760px){.rss-app{padding:0}.shell{border-radius:0;border-left:0;border-right:0}.topbar{height:auto;align-items:flex-start;flex-wrap:wrap;padding:12px}.brand{min-width:100%;}.top-actions{margin-left:0;width:100%;justify-content:space-between}.main{display:block}.panel{border-right:0;border-bottom:1px solid var(--line)}.route-list{max-height:none}.canvas{grid-template-columns:1fr;padding:12px}.steps{grid-template-columns:1fr 1fr;height:auto}.step{height:54px}.sub-body{display:block}.detect-row,.validation,.mapping-grid{grid-template-columns:1fr}.tabs{width:100%;overflow:auto}.tab{white-space:nowrap}}
</style>
</head>
<body>
<div class="rss-app">
  <div class="shell">
    <header class="topbar">
      <div class="brand"><div class="mark">R</div><div><div class="brand-title">RSS Source Manager</div><div class="brand-sub">Add direct feeds and RSSHub routes to a notebook workspace</div></div></div>
      <nav class="tabs" aria-label="RSS source mode">
        <button class="tab active" data-mode="url">Direct URL</button>
        <button class="tab" data-mode="rsshub">RSSHub route</button>
        <button class="tab" data-mode="keyword">Keyword discovery</button>
      </nav>
      <div class="top-actions"><span class="status-pill"><span class="dot"></span>rss datasource ready</span><button class="secondary">Test URL</button><button class="primary">Subscribe</button></div>
    </header>
    <main class="main">
      <aside class="panel">
        <div class="panel-head"><div><div class="panel-title">Discover source</div><div class="panel-note">Input is classified before preview</div></div><span class="count">3 paths</span></div>
        <div class="source-input">
          <div class="input-wrap"><input id="sourceInput" value="rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw" aria-label="Source URL or keyword"><button class="secondary" id="detectBtn">Detect</button></div>
          <div class="detect-row">
            <div class="detect" data-detect="url"><b>Direct RSS</b><span>http or https feed URL</span></div>
            <div class="detect active" data-detect="rsshub"><b>RSSHub</b><span>rsshub:// route gateway</span></div>
            <div class="detect" data-detect="keyword"><b>Search</b><span>plain text discovery</span></div>
          </div>
        </div>
        <div class="categories"><button class="chip active">Popular</button><button class="chip">Video</button><button class="chip">Programming</button><button class="chip">Social</button><button class="chip">News</button><button class="chip">Finance</button></div>
        <div class="route-list" id="routeList">
          <button class="route-card active" data-name="YouTube Channel" data-route="youtube/channel/:id" data-url="rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw" data-category="Video" data-view="Videos" data-heat="278k"><div class="route-top"><div><div class="route-name">YouTube Channel</div><div class="route-path">youtube/channel/:id</div></div><span class="tag heat">278k</span></div><div class="route-meta"><span class="tag">video</span><span class="tag">parameterized</span><span class="tag">top feed</span></div></button>
          <button class="route-card" data-name="GitHub Issues" data-route="github/issue/:owner/:repo" data-url="rsshub://github/issue/spur-dev/spur" data-category="Programming" data-view="Articles" data-heat="92k"><div class="route-top"><div><div class="route-name">GitHub Issues</div><div class="route-path">github/issue/:owner/:repo</div></div><span class="tag heat">92k</span></div><div class="route-meta"><span class="tag">programming</span><span class="tag">issues</span><span class="tag">radar</span></div></button>
          <button class="route-card" data-name="Reddit Subreddit" data-route="reddit/subreddit/:name" data-url="rsshub://reddit/subreddit/rust" data-category="Social" data-view="Social" data-heat="184k"><div class="route-top"><div><div class="route-name">Reddit Subreddit</div><div class="route-path">reddit/subreddit/:name</div></div><span class="tag heat">184k</span></div><div class="route-meta"><span class="tag">social</span><span class="tag">community</span></div></button>
          <button class="route-card" data-name="Hacker News Jobs" data-route="hackernews/jobs" data-url="rsshub://hackernews/jobs" data-category="News" data-view="Articles" data-heat="61k"><div class="route-top"><div><div class="route-name">Hacker News Jobs</div><div class="route-path">hackernews/jobs</div></div><span class="tag heat">61k</span></div><div class="route-meta"><span class="tag">news</span><span class="tag">no params</span></div></button>
        </div>
      </aside>
      <section class="workflow">
        <div class="steps"><div class="step done"><div class="step-num">1</div><div class="step-text"><div class="step-label">Detect</div><div class="step-detail">URL, route, or keyword</div></div></div><div class="step active"><div class="step-num">2</div><div class="step-text"><div class="step-label">Parameterize</div><div class="step-detail">Fill route variables</div></div></div><div class="step"><div class="step-num">3</div><div class="step-text"><div class="step-label">Preview</div><div class="step-detail">Fetch feed metadata</div></div></div><div class="step"><div class="step-num">4</div><div class="step-text"><div class="step-label">Subscribe</div><div class="step-detail">Save user settings</div></div></div></div>
        <div class="canvas">
          <section class="section"><div class="section-head"><div class="section-title">Route setup</div><span class="tag" id="routeCategory">Video</span></div><div class="section-body">
            <div class="field"><label>Selected source</label><input id="selectedName" value="YouTube Channel"></div>
            <div class="field"><label>Route template</label><input class="mono" id="routeTemplate" value="youtube/channel/:id"></div>
            <div class="field"><label>Parameter: id</label><input class="mono" id="routeParam" value="UCYO_jab_esuFRV4b17AJtAw"></div>
            <div class="field"><label>Generated feed URL</label><div class="url-box" id="generatedUrl">rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw</div></div>
            <div class="validation"><div class="metric"><b class="ok">Resolved</b><span>Protocol accepted</span></div><div class="metric"><b id="heatMetric">278k</b><span>RSSHub heat</span></div><div class="metric"><b>12h</b><span>Route cache TTL</span></div></div>
          </div></section>
          <section class="section"><div class="section-head"><div class="section-title">Feed preview</div><span class="tag">rss_feed(url)</span></div><div class="section-body">
            <div class="preview-card"><div class="feed-head"><div class="favicon">YT</div><div><div class="feed-title" id="previewTitle">3Blue1Brown uploads</div><div class="feed-desc">Validated through RSSHub. Latest entries are fetched with rss_entries(url) before subscription is saved.</div></div></div><div class="entry"><div class="entry-title">But what is a convolution?</div><div class="entry-meta">2 days ago - video - generated from route output</div></div><div class="entry"><div class="entry-title">A visual guide to transformers</div><div class="entry-meta">1 week ago - video - includes canonical link</div></div><div class="entry"><div class="entry-title">Neural networks, memorization, and grokking</div><div class="entry-meta">3 weeks ago - video - author preserved</div></div></div>
          </div></section>
          <section class="section"><div class="section-head"><div class="section-title">Storage mapping</div><span class="tag">subscription layer</span></div><div class="section-body"><div class="mapping-grid"><div class="map-item"><div class="map-label">feeds.url</div><div class="map-value mono" id="mapUrl">rsshub://youtube/channel/...</div></div><div class="map-item"><div class="map-label">subscriptions.view</div><div class="map-value" id="mapView">Videos</div></div><div class="map-item"><div class="map-label">subscriptions.category</div><div class="map-value" id="mapCategory">Video</div></div><div class="map-item"><div class="map-label">entries source</div><div class="map-value mono">rss_entries(url)</div></div></div></div></section>
        </div>
      </section>
      <aside class="panel subscribe"><div class="panel-head"><div><div class="panel-title">Subscribe</div><div class="panel-note">User-owned subscription settings</div></div><span class="count">ready</span></div><div class="sub-body"><div><div class="summary"><div class="summary-row"><span>Feed</span><span id="sumFeed">3Blue1Brown uploads</span></div><div class="summary-row"><span>Source</span><span id="sumSource">rsshub:// route</span></div><div class="summary-row"><span>View</span><span id="sumView">Videos</span></div><div class="summary-row"><span>Folder</span><span id="sumFolder">Video</span></div></div><div class="field"><label>Display title</label><input id="displayTitle" value="3Blue1Brown uploads"></div><div class="field"><label>Folder</label><select id="folderSelect"><option>Video</option><option>Programming</option><option>Research</option><option>News</option><option>Inbox</option></select></div><div class="toggle-list"><div class="toggle"><div class="toggle-text"><b>Show in main timeline</b><span>Entries appear in the default notebook feed view</span></div><span class="switch on"></span></div><div class="toggle"><div class="toggle-text"><b>Private subscription</b><span>Hide from shared source lists</span></div><span class="switch"></span></div></div><div class="health" id="healthBox">Feed validated. 3 recent entries found, channel metadata parsed, no RSSHub config requirement detected.</div><div class="action-stack"><button class="primary">Subscribe to feed</button><button class="secondary">Create notebook query cell</button></div></div><div class="implementation"><h4>Implementation fit</h4><div class="impl-row"><b>Discovery</b><code>rss_routes</code></div><div class="impl-row"><b>Preview</b><code>rss_feed(url), rss_entries(url)</code></div><div class="impl-row"><b>Protocol</b><code>rsshub:// resolves through configured base</code></div><div class="impl-row"><b>Persist</b><code>feeds + subscriptions + entries tables</code></div><div class="activity"><h4>Activity</h4><div class="event"><span class="event-dot"></span><div><b>Input classified as RSSHub</b><span>Route browser selected a parameterized source.</span></div></div><div class="event"><span class="event-dot"></span><div><b>Preview fetched</b><span>Gateway can show channel metadata before saving.</span></div></div></div></div></div></aside>
    </main>
  </div>
</div>
<script>
(function(){
  var cards = Array.prototype.slice.call(document.querySelectorAll('.route-card'));
  var input = document.getElementById('sourceInput');
  var detectEls = Array.prototype.slice.call(document.querySelectorAll('.detect'));
  var previewTitles = {
    'YouTube Channel':'3Blue1Brown uploads',
    'GitHub Issues':'SPUR issue tracker',
    'Reddit Subreddit':'r/rust community posts',
    'Hacker News Jobs':'Hacker News jobs'
  };
  function classify(value){
    var v = (value || '').trim();
    if(/^rsshub:\/\//i.test(v)) return 'rsshub';
    if(/^https?:\/\//i.test(v)) return 'url';
    return 'keyword';
  }
  function setDetect(kind){
    detectEls.forEach(function(el){ el.classList.toggle('active', el.getAttribute('data-detect') === kind); });
  }
  function updateFromCard(card){
    cards.forEach(function(c){ c.classList.toggle('active', c === card); });
    var name = card.getAttribute('data-name');
    var route = card.getAttribute('data-route');
    var url = card.getAttribute('data-url');
    var category = card.getAttribute('data-category');
    var view = card.getAttribute('data-view');
    var heat = card.getAttribute('data-heat');
    input.value = url;
    document.getElementById('selectedName').value = name;
    document.getElementById('routeTemplate').value = route;
    document.getElementById('generatedUrl').textContent = url;
    document.getElementById('routeCategory').textContent = category;
    document.getElementById('heatMetric').textContent = heat;
    document.getElementById('previewTitle').textContent = previewTitles[name] || name;
    document.getElementById('displayTitle').value = previewTitles[name] || name;
    document.getElementById('sumFeed').textContent = previewTitles[name] || name;
    document.getElementById('sumView').textContent = view;
    document.getElementById('sumFolder').textContent = category;
    document.getElementById('mapView').textContent = view;
    document.getElementById('mapCategory').textContent = category;
    document.getElementById('mapUrl').textContent = url.length > 32 ? url.slice(0, 29) + '...' : url;
    document.getElementById('folderSelect').value = category;
    setDetect(classify(url));
  }
  cards.forEach(function(card){ card.addEventListener('click', function(){ updateFromCard(card); }); });
  document.getElementById('detectBtn').addEventListener('click', function(){ setDetect(classify(input.value)); });
  input.addEventListener('input', function(){
    var kind = classify(input.value);
    setDetect(kind);
    document.getElementById('generatedUrl').textContent = input.value || 'Enter a source to generate a feed URL';
    document.getElementById('sumSource').textContent = kind === 'rsshub' ? 'rsshub:// route' : kind === 'url' ? 'direct RSS URL' : 'keyword search';
  });
  Array.prototype.slice.call(document.querySelectorAll('.tab')).forEach(function(tab){
    tab.addEventListener('click', function(){
      document.querySelectorAll('.tab').forEach(function(t){ t.classList.remove('active'); });
      tab.classList.add('active');
      var mode = tab.getAttribute('data-mode');
      if(mode === 'url') input.value = 'https://example.com/feed.xml';
      if(mode === 'rsshub') input.value = document.querySelector('.route-card.active').getAttribute('data-url');
      if(mode === 'keyword') input.value = 'machine learning video lectures';
      input.dispatchEvent(new Event('input'));
    });
  });
})();
</script>
</body>
</html>`;
await Deno.jupyter.display({ "text/html": html }, { raw: true });

<!doctype html>
 
 
 
 

 
 
 
 
 
 R RSS Source Manager Add direct feeds and RSSHub routes to a notebook workspace 
 
 Direct URL 
 RSSHub route 
 Keyword discovery 
 
 rss datasource ready Test URL Subscribe 
 
 
 
 Discover source Input is classified before preview 3 paths 
 
 Detect 
 
 Direct RSS http or https feed URL 
 RSSHub rsshub:// route gateway 
 Search plain text discovery 
 
 
 Popular Video Programming Social News Finance 
 
 YouTube Channel youtube/channel/:id 278k video parameterized top feed 
 GitHub Issues github/issue/:owner/:repo 92k programming issues radar 
 Reddit Subreddit reddit/subreddit/:name 184k social community 
 Hacker News Jobs hackernews/jobs 61k news no params 
 
 
 
 1 Detect URL, route, or keyword 2 Parameterize Fill route variables 3 Preview Fetch feed metadata 4 Subscribe Save user settings 
 
 Route setup Video 
 Selected source 
 Route template 
 Parameter: id 
 Generated feed URL rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw 
 Resolved Protocol accepted 278k RSSHub heat 12h Route cache TTL 
 
 Feed preview rss_feed(url) 
 YT 3Blue1Brown uploads Validated through RSSHub. Latest entries are fetched with rss_entries(url) before subscription is saved. But what is a convolution? 2 days ago - video - generated from route output A visual guide to transformers 1 week ago - video - includes canonical link Neural networks, memorization, and grokking 3 weeks ago - video - author preserved 
 
 Storage mapping subscription layer feeds.url rsshub://youtube/channel/... subscriptions.view Videos subscriptions.category Video entries source rss_entries(url) 
 
 
 Subscribe User-owned subscription settings ready Feed 3Blue1Brown uploads Source rsshub:// route View Videos Folder Video Display title Folder Video Programming Research News Inbox Show in main timeline Entries appear in the default notebook feed view Private subscription Hide from shared source lists Feed validated. 3 recent entries found, channel metadata parsed, no RSSHub config requirement detected. Subscribe to feed Create notebook query cell Implementation fit Discovery rss_routes Preview rss_feed(url), rss_entries(url) Protocol rsshub:// resolves through configured base Persist feeds + subscriptions + entries tables Activity Input classified as RSSHub Route browser selected a parameterized source. Preview fetched Gateway can show channel metadata before saving.

# RSS / RSSHub Design Critique

What works:
- The workflow matches the research doc: direct RSS URL, RSSHub route, and keyword discovery all converge into preview and subscription.
- The artifact maps cleanly to existing gateway primitives: `rss_routes`, `rss_feed(url)`, and `rss_entries(url)`.
- The UI keeps subscription settings separate from route discovery, which should reduce accidental subscriptions.

Revision needed:
- The first artifact defaulted the top mode tab to Direct URL while the body showed an RSSHub route. The v2 artifact below makes RSSHub the initial selected mode and tightens the layout around the subscription decision.

In [3]:
// open-design artifact v2: corrected RSSHub-first subscription UI
const html = String.raw`<!doctype html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<style>
*{box-sizing:border-box}body{margin:0;background:#f4f1eb;color:#171717;font-family:Inter,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;letter-spacing:0}.wrap{min-height:100vh;padding:18px}.app{max-width:1360px;margin:auto;background:#fff;border:1px solid #d8d2c7;border-radius:8px;overflow:hidden;box-shadow:0 18px 42px rgba(47,39,25,.14)}.bar{display:flex;align-items:center;gap:12px;padding:12px 14px;border-bottom:1px solid #d8d2c7;background:#fbfaf7}.logo{width:30px;height:30px;border-radius:6px;background:#f0bd43;border:1px solid #111;display:grid;place-items:center;font-weight:800}.title{font-size:14px;font-weight:800}.sub{font-size:12px;color:#6b6b6b;margin-top:2px}.modes{display:flex;gap:4px;border:1px solid #d8d2c7;border-radius:7px;padding:3px;background:#fff;margin-left:18px}.mode{height:30px;border:0;border-radius:5px;background:transparent;padding:0 10px;font-size:12px;color:#555}.mode.active{background:#151515;color:#fff}.bar-actions{margin-left:auto;display:flex;gap:8px;align-items:center}.pill{height:28px;border:1px solid #d8d2c7;border-radius:999px;padding:0 10px;display:inline-flex;align-items:center;gap:7px;font-size:12px;color:#555;background:#fff}.dot{width:7px;height:7px;border-radius:50%;background:#168052}.btn{height:32px;border-radius:6px;border:1px solid #151515;background:#151515;color:#fff;padding:0 12px;font-size:12px;font-weight:700}.btn.secondary{background:#fff;color:#171717;border-color:#bcb5aa}.grid{display:grid;grid-template-columns:310px 1fr 350px;min-height:720px}.col{border-right:1px solid #d8d2c7;min-width:0}.col:last-child{border-right:0}.head{height:62px;padding:12px 14px;border-bottom:1px solid #d8d2c7;background:#fbfaf7;display:flex;justify-content:space-between;align-items:center}.h{font-size:13px;font-weight:800}.hint{font-size:11px;color:#707070;margin-top:3px}.count{font-size:11px;border:1px solid #d8d2c7;border-radius:999px;padding:4px 8px;background:#fff;color:#555}.input{padding:14px;border-bottom:1px solid #d8d2c7}.input input{width:100%;height:38px;border:1px solid #bcb5aa;border-radius:6px;padding:0 10px;font-size:13px;font-family:"SF Mono",Consolas,monospace}.detect{display:grid;grid-template-columns:1fr 1fr 1fr;gap:6px;margin-top:10px}.d{border:1px solid #d8d2c7;border-radius:6px;padding:8px;background:#fbfaf7;min-height:58px}.d.active{border-color:#10766b;background:#dff1ed}.d b{display:block;font-size:11px}.d span{display:block;font-size:11px;color:#5f5f5f;margin-top:4px;line-height:1.25}.chips{display:flex;flex-wrap:wrap;gap:6px;padding:10px}.chip{height:26px;border:1px solid #d8d2c7;border-radius:999px;background:#fff;padding:0 9px;font-size:11px;color:#4f4f4f}.chip.active{background:#171717;color:#fff;border-color:#171717}.routes{padding:0 10px 12px;display:grid;gap:8px}.route{border:1px solid #d8d2c7;border-radius:8px;background:#fff;padding:10px;text-align:left}.route.active{border-color:#10766b;box-shadow:inset 3px 0 0 #10766b}.rtop{display:flex;justify-content:space-between;gap:8px}.rname{font-size:13px;font-weight:800}.rpath{font-family:"SF Mono",Consolas,monospace;font-size:11px;color:#555;margin-top:4px;word-break:break-all}.tag{font-size:10px;border:1px solid #d8d2c7;border-radius:999px;padding:3px 6px;background:#fafafa;color:#555}.heat{background:#fff0cb;color:#8a5200;border-color:#e2c16d}.route-tags{display:flex;flex-wrap:wrap;gap:6px;margin-top:8px}.steps{display:grid;grid-template-columns:repeat(4,1fr);height:62px;border-bottom:1px solid #d8d2c7;background:#fbfaf7}.step{display:flex;gap:8px;align-items:center;padding:0 12px;border-right:1px solid #d8d2c7;min-width:0}.step:last-child{border-right:0}.num{width:24px;height:24px;border-radius:6px;border:1px solid #d8d2c7;background:#eee;display:grid;place-items:center;font-size:11px;font-weight:800}.step.done .num{background:#dcf1e7;color:#16794c;border-color:#afd9c2}.step.active .num{background:#171717;color:#fff;border-color:#171717}.slabel{font-size:11px;font-weight:800;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.sdetail{font-size:10px;color:#777;margin-top:2px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.work{background:#fdfbf7}.canvas{padding:16px;display:grid;grid-template-columns:1fr 1fr;gap:14px}.section{border:1px solid #d8d2c7;border-radius:8px;background:#fff;overflow:hidden}.section.wide{grid-column:1/-1}.shead{height:43px;padding:0 12px;border-bottom:1px solid #d8d2c7;background:#fbfaf7;display:flex;align-items:center;justify-content:space-between}.stitle{font-size:13px;font-weight:800}.body{padding:13px}.field{display:grid;gap:6px;margin-bottom:11px}.field label{font-size:11px;font-weight:800;color:#4b4b4b}.field input,.field select{height:36px;border:1px solid #bcb5aa;border-radius:6px;padding:0 9px;font-size:13px;background:#fff}.mono{font-family:"SF Mono",Consolas,monospace}.url{border:1px solid #d8d2c7;border-radius:6px;background:#f7f5ef;padding:10px;font:12px/1.45 "SF Mono",Consolas,monospace;word-break:break-all}.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:8px}.metric{border:1px solid #d8d2c7;border-radius:7px;padding:9px;background:#fbfaf7}.metric b{font-size:13px}.metric span{display:block;font-size:10px;color:#777;margin-top:3px}.ok{color:#16794c}.feed{border:1px solid #d8d2c7;border-radius:8px;overflow:hidden}.feed-head{display:flex;gap:10px;padding:12px;border-bottom:1px solid #d8d2c7}.avatar{width:42px;height:42px;border-radius:8px;background:linear-gradient(135deg,#f0bd43,#37a99a);display:grid;place-items:center;font-weight:900}.feed-title{font-size:14px;font-weight:850}.feed-desc{font-size:12px;color:#666;line-height:1.35;margin-top:4px}.entry{padding:10px 12px;border-top:1px solid #d8d2c7}.entry:first-of-type{border-top:0}.entry b{display:block;font-size:12px}.entry span{display:block;font-size:11px;color:#777;margin-top:4px}.map{display:grid;grid-template-columns:repeat(4,1fr);gap:8px}.map div{border:1px solid #d8d2c7;border-radius:7px;padding:9px;background:#fbfaf7}.map small{display:block;color:#666;text-transform:uppercase;font-size:10px}.map b{display:block;font-size:12px;margin-top:5px}.side{background:#fbfaf7}.sidebody{padding:14px}.summary{border:1px solid #d8d2c7;border-radius:8px;background:#fff;overflow:hidden}.row{display:flex;justify-content:space-between;gap:12px;padding:10px 12px;border-top:1px solid #d8d2c7;font-size:12px}.row:first-child{border-top:0}.row span:first-child{color:#666}.row span:last-child{text-align:right;font-weight:750}.toggle{display:flex;justify-content:space-between;gap:12px;align-items:center;border:1px solid #d8d2c7;border-radius:7px;background:#fff;padding:10px;margin-top:8px}.toggle b{display:block;font-size:12px}.toggle span{display:block;font-size:11px;color:#666;margin-top:3px}.switch{width:36px;height:20px;border-radius:999px;background:#10766b;position:relative;flex:0 0 auto}.switch:after{content:"";position:absolute;right:2px;top:2px;width:16px;height:16px;background:#fff;border-radius:50%;box-shadow:0 1px 3px rgba(0,0,0,.2)}.health{margin-top:12px;border:1px solid #b2d9c2;background:#dcf1e7;color:#115a39;border-radius:8px;padding:10px;font-size:12px;line-height:1.4}.actions{display:grid;gap:8px;margin-top:12px}.actions .btn{height:38px}.impl{margin-top:14px;border:1px solid #d8d2c7;background:#fff;border-radius:8px;padding:12px}.impl h4{margin:0 0 8px;font-size:12px}.implrow{display:grid;grid-template-columns:96px 1fr;gap:8px;border-top:1px solid #eee;padding:6px 0;font-size:11px}.implrow:first-of-type{border-top:0}.impl code{font-family:"SF Mono",Consolas,monospace;word-break:break-word}@media(max-width:1080px){.grid{grid-template-columns:300px 1fr}.side{grid-column:1/-1;border-top:1px solid #d8d2c7}.sidebody{display:grid;grid-template-columns:1fr 1fr;gap:14px}.impl{margin-top:0}}@media(max-width:760px){.wrap{padding:0}.app{border-radius:0;border-left:0;border-right:0}.bar{flex-wrap:wrap}.modes{margin-left:0;width:100%;overflow:auto}.bar-actions{margin-left:0;width:100%;justify-content:space-between}.grid{display:block}.col{border-right:0;border-bottom:1px solid #d8d2c7}.steps{grid-template-columns:1fr 1fr;height:auto}.step{height:54px}.canvas{grid-template-columns:1fr;padding:12px}.map,.metrics,.detect{grid-template-columns:1fr}.sidebody{display:block}}
</style></head><body><div class="wrap"><div class="app"><header class="bar"><div class="logo">R</div><div><div class="title">RSS Source Manager</div><div class="sub">Subscribe to direct RSS feeds or RSSHub routes from the notebook</div></div><nav class="modes"><button class="mode">Direct URL</button><button class="mode active">RSSHub route</button><button class="mode">Keyword discovery</button></nav><div class="bar-actions"><span class="pill"><span class="dot"></span>rss datasource ready</span><button class="btn secondary">Test URL</button><button class="btn">Subscribe</button></div></header><main class="grid"><aside class="col"><div class="head"><div><div class="h">Discover source</div><div class="hint">Classify input before preview</div></div><span class="count">3 paths</span></div><div class="input"><input value="rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw"><div class="detect"><div class="d"><b>Direct RSS</b><span>http or https feed URL</span></div><div class="d active"><b>RSSHub</b><span>rsshub:// route gateway</span></div><div class="d"><b>Search</b><span>plain text discovery</span></div></div></div><div class="chips"><button class="chip active">Popular</button><button class="chip">Video</button><button class="chip">Programming</button><button class="chip">Social</button><button class="chip">News</button></div><div class="routes"><button class="route active"><div class="rtop"><div><div class="rname">YouTube Channel</div><div class="rpath">youtube/channel/:id</div></div><span class="tag heat">278k</span></div><div class="route-tags"><span class="tag">video</span><span class="tag">parameterized</span><span class="tag">top feed</span></div></button><button class="route"><div class="rtop"><div><div class="rname">GitHub Issues</div><div class="rpath">github/issue/:owner/:repo</div></div><span class="tag heat">92k</span></div><div class="route-tags"><span class="tag">programming</span><span class="tag">issues</span></div></button><button class="route"><div class="rtop"><div><div class="rname">Reddit Subreddit</div><div class="rpath">reddit/subreddit/:name</div></div><span class="tag heat">184k</span></div><div class="route-tags"><span class="tag">social</span><span class="tag">community</span></div></button></div></aside><section class="col work"><div class="steps"><div class="step done"><div class="num">1</div><div><div class="slabel">Detect</div><div class="sdetail">route selected</div></div></div><div class="step active"><div class="num">2</div><div><div class="slabel">Parameterize</div><div class="sdetail">fill variables</div></div></div><div class="step"><div class="num">3</div><div><div class="slabel">Preview</div><div class="sdetail">fetch metadata</div></div></div><div class="step"><div class="num">4</div><div><div class="slabel">Subscribe</div><div class="sdetail">save settings</div></div></div></div><div class="canvas"><section class="section"><div class="shead"><div class="stitle">Route setup</div><span class="tag">Video</span></div><div class="body"><div class="field"><label>Route template</label><input class="mono" value="youtube/channel/:id"></div><div class="field"><label>Parameter: id</label><input class="mono" value="UCYO_jab_esuFRV4b17AJtAw"></div><div class="field"><label>Generated feed URL</label><div class="url">rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw</div></div><div class="metrics"><div class="metric"><b class="ok">Resolved</b><span>protocol accepted</span></div><div class="metric"><b>278k</b><span>RSSHub heat</span></div><div class="metric"><b>12h</b><span>route cache TTL</span></div></div></div></section><section class="section"><div class="shead"><div class="stitle">Feed preview</div><span class="tag">rss_feed(url)</span></div><div class="body"><div class="feed"><div class="feed-head"><div class="avatar">YT</div><div><div class="feed-title">3Blue1Brown uploads</div><div class="feed-desc">Validated through RSSHub. Entries are sampled before the subscription is persisted.</div></div></div><div class="entry"><b>But what is a convolution?</b><span>2 days ago - video - canonical link found</span></div><div class="entry"><b>A visual guide to transformers</b><span>1 week ago - author and categories parsed</span></div><div class="entry"><b>Neural networks and grokking</b><span>3 weeks ago - media metadata preserved</span></div></div></div></section><section class="section wide"><div class="shead"><div class="stitle">Storage mapping</div><span class="tag">subscription layer</span></div><div class="body"><div class="map"><div><small>feeds.url</small><b class="mono">rsshub://youtube/...</b></div><div><small>subscriptions.view</small><b>Videos</b></div><div><small>subscriptions.category</small><b>Video</b></div><div><small>entries source</small><b class="mono">rss_entries(url)</b></div></div></div></section></div></section><aside class="col side"><div class="head"><div><div class="h">Subscribe</div><div class="hint">User-owned feed settings</div></div><span class="count">ready</span></div><div class="sidebody"><div><div class="summary"><div class="row"><span>Feed</span><span>3Blue1Brown uploads</span></div><div class="row"><span>Source</span><span>rsshub:// route</span></div><div class="row"><span>View</span><span>Videos</span></div><div class="row"><span>Folder</span><span>Video</span></div></div><div class="field" style="margin-top:12px"><label>Display title</label><input value="3Blue1Brown uploads"></div><div class="field"><label>Folder</label><select><option>Video</option><option>Programming</option><option>Research</option><option>Inbox</option></select></div><div class="toggle"><div><b>Show in main timeline</b><span>Entries appear in the default notebook feed view</span></div><i class="switch"></i></div><div class="health">Feed validated. 3 recent entries found, channel metadata parsed, no RSSHub config requirement detected.</div><div class="actions"><button class="btn">Subscribe to feed</button><button class="btn secondary">Create notebook query cell</button></div></div><div class="impl"><h4>Implementation fit</h4><div class="implrow"><b>Discovery</b><code>rss_routes</code></div><div class="implrow"><b>Preview</b><code>rss_feed(url), rss_entries(url)</code></div><div class="implrow"><b>Protocol</b><code>rsshub:// resolves through configured base</code></div><div class="implrow"><b>Persist</b><code>feeds + subscriptions + entries</code></div></div></div></aside></main></div></div></body></html>`;
await Deno.jupyter.display({ "text/html": html }, { raw: true });

<!doctype html>
 
 R RSS Source Manager Subscribe to direct RSS feeds or RSSHub routes from the notebook Direct URL RSSHub route Keyword discovery rss datasource ready Test URL Subscribe Discover source Classify input before preview 3 paths Direct RSS http or https feed URL RSSHub rsshub:// route gateway Search plain text discovery Popular Video Programming Social News YouTube Channel youtube/channel/:id 278k video parameterized top feed GitHub Issues github/issue/:owner/:repo 92k programming issues Reddit Subreddit reddit/subreddit/:name 184k social community 1 Detect route selected 2 Parameterize fill variables 3 Preview fetch metadata 4 Subscribe save settings Route setup Video Route template Parameter: id Generated feed URL rsshub://youtube/channel/UCYO_jab_esuFRV4b17AJtAw Resolved protocol accepted 278k RSSHub heat 12h route cache TTL Feed preview rss_feed(url) YT 3Blue1Brown uploads Validated through RSSHub. Entries are sampled before the subscription is persisted. But what is a convolution? 2 days ago - video - canonical link found A visual guide to transformers 1 week ago - author and categories parsed Neural networks and grokking 3 weeks ago - media metadata preserved Storage mapping subscription layer feeds.url rsshub://youtube/... subscriptions.view Videos subscriptions.category Video entries source rss_entries(url) Subscribe User-owned feed settings ready Feed 3Blue1Brown uploads Source rsshub:// route View Videos Folder Video Display title Folder Video Programming Research Inbox Show in main timeline Entries appear in the default notebook feed view Feed validated. 3 recent entries found, channel metadata parsed, no RSSHub config requirement detected. Subscribe to feed Create notebook query cell Implementation fit Discovery rss_routes Preview rss_feed(url), rss_entries(url) Protocol rsshub:// resolves through configured base Persist feeds + subscriptions + entries